# Stage 1 - Data rescue and cleaning

Cleans the four raw files in `data/raw/` and writes analytics-ready versions to `data/cleaned/`. Row counts and flag counts per file go to `data/cleaned/cleaning_log.txt`.

In [1]:
import json, re
from pathlib import Path
import pandas as pd
from utils import *

RAW = Path("../data/raw")
OUT = Path("../data/cleaned")
OUT.mkdir(exist_ok=True)

log_lines = []
def log(msg=""):
    print(msg)
    log_lines.append(msg)

def read_csv_raw(name):
    return pd.read_csv(RAW / name, dtype=str, keep_default_na=False)

# keep the "best" record when one id maps to several conflicting rows: fewest nulls,
# then the latest date column. Conflicts are written to a side file rather than lost.
def resolve_id_conflicts(df, key, date_col, name):
    dup_mask = df[key].duplicated(keep=False)
    if not dup_mask.any():
        return df
    ranked = df.assign(_nulls=df.isna().sum(axis=1)).sort_values([key, "_nulls", date_col], ascending=[True, True, False])
    keep = ~ranked[key].duplicated()
    conflicts = ranked[~keep].drop(columns="_nulls")
    conflicts.to_csv(OUT / f"{name}_id_conflicts.csv", index=False)
    log(f"  {key} collisions with differing content: {dup_mask.sum()} rows across {df.loc[dup_mask, key].nunique()} ids -> kept best record, {len(conflicts)} rows moved to {name}_id_conflicts.csv")
    return ranked[keep].drop(columns="_nulls").reset_index(drop=True)

## 1. UPI transactions

In [2]:
tx_raw = read_csv_raw("track1_upi_transactions.csv")
log("=== upi_transactions ===")
log(f"raw rows: {len(tx_raw)}")

tx = pd.DataFrame({
    "txn_id": tx_raw.txn_id.map(lambda v: normalize_id(v, "TXN", 8)),
    "timestamp": tx_raw.timestamp.map(parse_flexible_date),
    "user_id": tx_raw.user_id.map(lambda v: normalize_id(v, "USR", 5)),
    "merchant_id": tx_raw.merchant_id.map(lambda v: normalize_id(v, "MCH", 4)),
    "amount": tx_raw.amount.map(parse_amount),
    "utr": tx_raw.utr.str.replace(r"[\s\-]", "", regex=True).str.upper().replace("", None),
    "mcc": tx_raw.mcc.map(normalize_mcc),
    "status": tx_raw.status.map(lambda v: normalize_status(v, TXN_STATUS)),
})

# negative amounts are kept and flagged - they could be reversals and matter for fraud analysis
tx["is_negative_flag"] = tx.amount < 0
tx["amount_missing"] = tx.amount.isna()
tx["timestamp_missing"] = tx.timestamp.isna()
tx["utr_valid"] = tx.utr.str.fullmatch(r"UTR\d{10}").fillna(False)
tx["mcc_missing"] = tx.mcc.isna()

unmapped = unmapped_values(tx_raw.status, tx.status)
log(f"status values not in mapping: {unmapped}")

before = len(tx)
tx = tx.drop_duplicates().reset_index(drop=True)
log(f"exact duplicate rows removed (post-normalization): {before - len(tx)}")
tx["txn_id_duplicate"] = tx.txn_id.duplicated(keep=False)
log(f"txn_id still duplicated with differing content: {tx.txn_id_duplicate.sum()}")

log(f"cleaned rows: {len(tx)}")
for c in ["is_negative_flag", "amount_missing", "timestamp_missing", "mcc_missing"]:
    log(f"  flagged {c}: {tx[c].sum()}")
log(f"  flagged utr_valid=False: {(~tx.utr_valid).sum()}  (of which utr missing: {tx.utr.isna().sum()})")
log(f"  status distribution: {tx.status.value_counts(dropna=False).to_dict()}")
log(f"  timestamp range: {tx.timestamp.min()} -> {tx.timestamp.max()}")

tx.to_csv(OUT / "upi_transactions_clean.csv", index=False)
tx.head()

=== upi_transactions ===
raw rows: 20400


status values not in mapping: {}
exact duplicate rows removed (post-normalization): 400
txn_id still duplicated with differing content: 0
cleaned rows: 20000
  flagged is_negative_flag: 420
  flagged amount_missing: 0
  flagged timestamp_missing: 0
  flagged mcc_missing: 2872
  flagged utr_valid=False: 1000  (of which utr missing: 1000)
  status distribution: {'SUCCESS': 17053, 'FAILED': 1955, 'PENDING': 992}
  timestamp range: 2026-01-01 00:00:00 -> 2026-03-31 23:53:53


,txn_id,timestamp,user_id,merchant_id,amount,utr,mcc,status,is_negative_flag,amount_missing,timestamp_missing,utr_valid,mcc_missing,txn_id_duplicate
0,TXN00011869,2026-01-15 00:11:30,USR45826,MCH7045,15722.34,UTR6498104698,5411,SUCCESS,False,False,False,True,False,False
1,TXN00010383,2026-01-17 20:09:44,USR79397,MCH5031,6362.90,UTR7656190355,4131,FAILED,False,False,False,True,False,False
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,5411,SUCCESS,False,False,False,True,False,False
3,TXN00006448,2026-02-02 20:17:51,USR54287,MCH6928,12110.49,UTR5257823698,5411,SUCCESS,False,False,False,True,False,False
4,TXN00018792,2026-03-31 14:02:37,USR53865,MCH8121,19432.94,UTR4204272894,4131,SUCCESS,False,False,False,True,False,False


## 2. KYC records

Aadhaar is never stored in full - only `aadhaar_last4` plus a validity flag survive.

In [3]:
kyc_raw = read_csv_raw("track1_kyc_records.csv")
log("")
log("=== kyc_records ===")
log(f"raw rows: {len(kyc_raw)}")

pan = kyc_raw.pan.str.replace(r"[\s\-]", "", regex=True).str.upper().replace("", None)
aadhaar_digits = kyc_raw.aadhaar.str.replace(r"[\s\-]", "", regex=True)

kyc = pd.DataFrame({
    "user_id": kyc_raw.user_id.map(lambda v: normalize_id(v, "USR", 5)),
    "full_name": kyc_raw.full_name.str.strip().str.title(),
    "pan": pan,
    "pan_valid": pan.str.fullmatch(r"[A-Z]{5}[0-9]{4}[A-Z]").fillna(False),
    "aadhaar_last4": aadhaar_digits.map(lambda s: re.sub(r"\D", "", s)[-4:] or None),
    # only a clean 12-digit number counts as valid; pre-masked XXXX-XXXX-1234 values keep
    # their last4 but cannot be verified, 10/13-digit values are just wrong
    "aadhaar_valid": aadhaar_digits.str.fullmatch(r"\d{12}").fillna(False),
    "aadhaar_premasked": aadhaar_digits.str.match(r"^X{8}\d{4}$", case=False),
    "date_of_birth": kyc_raw.date_of_birth.map(parse_flexible_date),
    "city": kyc_raw.city.map(normalize_city),
    "state": kyc_raw.state.str.strip().str.title(),
    "monthly_income": kyc_raw.monthly_income.map(parse_amount),
    "occupation": kyc_raw.occupation.str.strip().str.title(),
    "signup_timestamp": kyc_raw.signup_timestamp.map(parse_flexible_date),
    "kyc_status": kyc_raw.kyc_status.map(lambda v: normalize_status(v, KYC_STATUS)),
    "risk_segment": kyc_raw.risk_segment.map(lambda v: normalize_status(v, RISK_SEGMENT)),
})

# negative income has no legitimate meaning, unlike a negative transaction amount
kyc["income_valid"] = kyc.monthly_income.notna() & (kyc.monthly_income >= 0)
kyc["income_missing"] = kyc.monthly_income.isna()
kyc["dob_missing"] = kyc.date_of_birth.isna()
kyc["signup_missing"] = kyc.signup_timestamp.isna()

log(f"kyc_status values not in mapping: {unmapped_values(kyc_raw.kyc_status, kyc.kyc_status)}")
log(f"risk_segment values not in mapping: {unmapped_values(kyc_raw.risk_segment, kyc.risk_segment)}")
log(f"distinct cities after aliasing: {sorted(kyc.city.dropna().unique())}")

before = len(kyc)
kyc = kyc.drop_duplicates().reset_index(drop=True)
log(f"exact duplicate rows removed (post-normalization): {before - len(kyc)}")
kyc = resolve_id_conflicts(kyc, "user_id", "signup_timestamp", "kyc_records")

log(f"cleaned rows: {len(kyc)}  (unique user_id: {kyc.user_id.nunique()})")
log(f"  flagged pan_valid=False: {(~kyc.pan_valid).sum()}  (of which pan missing: {kyc.pan.isna().sum()})")
log(f"  flagged aadhaar_valid=False: {(~kyc.aadhaar_valid).sum()}  (pre-masked: {kyc.aadhaar_premasked.sum()}, no digits at all: {kyc.aadhaar_last4.isna().sum()})")
log(f"  flagged income_valid=False: {(~kyc.income_valid).sum()}  (negative: {(kyc.monthly_income < 0).sum()}, missing: {kyc.income_missing.sum()})")
log(f"  flagged dob_missing: {kyc.dob_missing.sum()}")
log(f"  flagged signup_missing: {kyc.signup_missing.sum()}")
log(f"  kyc_status distribution: {kyc.kyc_status.value_counts(dropna=False).to_dict()}")
log(f"  risk_segment distribution: {kyc.risk_segment.value_counts(dropna=False).to_dict()}")

kyc.to_csv(OUT / "kyc_records_clean.csv", index=False)
kyc.head()


=== kyc_records ===
raw rows: 36400


kyc_status values not in mapping: {}
risk_segment values not in mapping: {}
distinct cities after aliasing: ['Amritsar', 'Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Jalandhar', 'Kolkata', 'Lucknow', 'Ludhiana', 'Mumbai', 'Pune']
exact duplicate rows removed (post-normalization): 486
  user_id collisions with differing content: 12964 rows across 5970 ids -> kept best record, 6994 rows moved to kyc_records_id_conflicts.csv
cleaned rows: 28920  (unique user_id: 28920)
  flagged pan_valid=False: 3469  (of which pan missing: 1328)
  flagged aadhaar_valid=False: 6815  (pre-masked: 2322, no digits at all: 1831)
  flagged income_valid=False: 4216  (negative: 1210, missing: 3006)
  flagged dob_missing: 2061
  flagged signup_missing: 1932
  kyc_status distribution: {'VERIFIED': 22380, 'PENDING': 4184, 'REJECTED': 2356}
  risk_segment distribution: {'LOW': 16721, 'MEDIUM': 7799, 'HIGH': 2962, 'UNKNOWN': 1438}


,user_id,full_name,pan,pan_valid,aadhaar_last4,aadhaar_valid,aadhaar_premasked,date_of_birth,city,state,monthly_income,occupation,signup_timestamp,kyc_status,risk_segment,income_valid,income_missing,dob_missing,signup_missing
0,USR10001,Osha Dewan,XZDVU00501,False,8310,True,False,1964-03-24 00:00:00,Pune,Maharashtra,37535.0,Retired,2025-09-27 00:00:00,VERIFIED,MEDIUM,True,False,False,False
1,USR10007,Yutika Borra,NaN,False,9606,True,False,1963-09-27 11:39:00,Bengaluru,Karnataka,20393.0,Unemployed,2025-02-07 00:00:00,VERIFIED,MEDIUM,True,False,False,False
2,USR10010,Warda Dada,WJVZZ1464B,True,2042,True,False,1964-07-26 14:51:21,Jalandhar,Punjab,48376.0,Gig Worker,2024-07-26 09:55:29,PENDING,HIGH,True,False,False,False
3,USR10013,Azaan Nigam,LQQSW5207I,True,3345,True,False,1982-12-12 00:00:00,Pune,Maharashtra,NaN,Salaried,2025-10-09 00:00:00,VERIFIED,HIGH,False,True,False,False
4,USR10014,Nisha Patil,RPDIH8589M,True,9228,False,True,1987-08-29 17:03:46,Mumbai,Maharashtra,18633.0,Freelancer,2026-01-28 00:00:00,VERIFIED,MEDIUM,True,False,False,False


## 3. Merchants master

In [4]:
m_raw = read_csv_raw("track1_merchants_master.csv")
log("")
log("=== merchants_master ===")
log(f"raw rows: {len(m_raw)}")

merchants = pd.DataFrame({
    "merchant_id": m_raw.merchant_id.map(lambda v: normalize_id(v, "MCH", 4)),
    "merchant_name": m_raw.merchant_name.str.strip().str.replace(r"\s+", " ", regex=True),
    "mcc": m_raw.mcc.map(normalize_mcc),
    "merchant_category": m_raw.merchant_category.map(lambda v: normalize_status(v, MERCHANT_CATEGORY)),
    "business_type": m_raw.business_type.map(lambda v: normalize_status(v, BUSINESS_TYPE)),
    "city": m_raw.city.map(normalize_city),
    "state": m_raw.state.str.strip().str.title(),
    "onboarding_date": m_raw.onboarding_date.map(parse_flexible_date),
    "settlement_account_last4": m_raw.settlement_account.map(mask_last4),
    "merchant_status": m_raw.merchant_status.map(lambda v: normalize_status(v, MERCHANT_STATUS)),
    "declared_avg_ticket_size": m_raw.declared_avg_ticket_size.map(parse_amount),
})

# category -> MCC is ~90% consistent in the raw data (the rest is injected noise), so a missing or
# garbage MCC ("misc", "NA", "UNKNOWN") is backfilled from the category. Existing MCCs are never overwritten.
merchants["mcc_inferred"] = merchants.mcc.isna() & merchants.merchant_category.notna()
merchants.loc[merchants.mcc_inferred, "mcc"] = merchants.loc[merchants.mcc_inferred, "merchant_category"].map(CATEGORY_TO_MCC)
merchants["mcc_missing"] = merchants.mcc.isna()

# negative ticket size is flagged for review rather than nulled - could be a sign convention issue upstream
merchants["ticket_size_negative"] = merchants.declared_avg_ticket_size < 0
merchants["ticket_size_missing"] = merchants.declared_avg_ticket_size.isna()
merchants["settlement_account_missing"] = merchants.settlement_account_last4.isna()
merchants["onboarding_missing"] = merchants.onboarding_date.isna()

for col, mapping in [("merchant_status", MERCHANT_STATUS), ("business_type", BUSINESS_TYPE), ("merchant_category", MERCHANT_CATEGORY)]:
    log(f"{col} values not in mapping: {unmapped_values(m_raw[col], merchants[col])}")

before = len(merchants)
merchants = merchants.drop_duplicates().reset_index(drop=True)
log(f"exact duplicate rows removed (post-normalization): {before - len(merchants)}")
merchants = resolve_id_conflicts(merchants, "merchant_id", "onboarding_date", "merchants_master")

log(f"cleaned rows: {len(merchants)}  (unique merchant_id: {merchants.merchant_id.nunique()})")
for c in ["mcc_inferred", "mcc_missing", "ticket_size_negative", "ticket_size_missing", "settlement_account_missing", "onboarding_missing"]:
    log(f"  flagged {c}: {merchants[c].sum()}")
log(f"  merchant_status distribution: {merchants.merchant_status.value_counts(dropna=False).to_dict()}")
log(f"  merchant_category distribution: {merchants.merchant_category.value_counts(dropna=False).to_dict()}")

merchants.to_csv(OUT / "merchants_master_clean.csv", index=False)
merchants.head()


=== merchants_master ===
raw rows: 6210
merchant_status values not in mapping: {}
business_type values not in mapping: {}
merchant_category values not in mapping: {}
exact duplicate rows removed (post-normalization): 118
  merchant_id collisions with differing content: 3103 rows across 1354 ids -> kept best record, 1749 rows moved to merchants_master_id_conflicts.csv
cleaned rows: 4343  (unique merchant_id: 4343)
  flagged mcc_inferred: 464
  flagged mcc_missing: 0
  flagged ticket_size_negative: 358
  flagged ticket_size_missing: 204
  flagged settlement_account_missing: 1432
  flagged onboarding_missing: 263
  merchant_status distribution: {'ACTIVE': 3563, 'INACTIVE': 780}
  merchant_category distribution: {'APPAREL': 474, 'TRANSPORT': 463, 'DEPARTMENT_STORE': 458, 'HOTEL': 444, 'TELECOM': 433, 'MISC_RETAIL': 427, 'GROCERY': 421, 'PHARMACY': 419, 'BOOKS_STATIONERY': 406, 'RESTAURANT': 398}


,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state,onboarding_date,settlement_account_last4,merchant_status,declared_avg_ticket_size,mcc_inferred,mcc_missing,ticket_size_negative,ticket_size_missing,settlement_account_missing,onboarding_missing
0,MCH1001,Anand-Kothari,5411,GROCERY,PARTNERSHIP,Pune,Maharashtra,2026-02-17 06:54:00,XXXX6762,ACTIVE,332.55,True,False,False,False,False,False
1,MCH1002,"Apte, Saha and Edwin",5812,RESTAURANT,PRIVATE_LIMITED,Bengaluru,Karnataka,2026-01-07 09:01:00,XXXX8596,ACTIVE,1671.20,False,False,False,False,False,False
2,MCH1003,"Nayar, Batta and Barad",5699,APPAREL,PRIVATE_LIMITED,Chennai,Tamil Nadu,NaT,NaN,ACTIVE,792.91,False,False,False,False,True,True
3,MCH1004,Pillai-Mohan,4131,TRANSPORT,SOLE_PROPRIETOR,Lucknow,Uttar Pradesh,2023-04-08 00:00:00,XXXX4449,ACTIVE,342.70,False,False,False,False,False,False
4,MCH1005,SAGAR AND SONS,5912,PHARMACY,PARTNERSHIP,Ludhiana,Punjab,2024-02-06 00:00:00,XXXX9775,ACTIVE,967.53,False,False,False,False,False,False


## 4. Chargebacks

In [5]:
with open(RAW / "track1_chargebacks.json") as f:
    cb_raw = pd.DataFrame(json.load(f)).astype(str)
log("")
log("=== chargebacks ===")
log(f"raw rows: {len(cb_raw)}")

cb = pd.DataFrame({
    "complaint_id": cb_raw.complaint_id.map(lambda v: normalize_id(v, "CBK", 7)),
    "txn_id": cb_raw.txn_id.map(lambda v: normalize_id(v, "TXN", 8)),
    "user_id": cb_raw.user_id.map(lambda v: normalize_id(v, "USR", 5)),
    "merchant_id": cb_raw.merchant_id.map(lambda v: normalize_id(v, "MCH", 4)),
    "disputed_amount": cb_raw.disputed_amount.map(parse_amount),   # blank -> NaN, never 0
    "reason_code": cb_raw.reason_code.str.strip(),
    "reason_category": cb_raw.reason_code.map(lambda v: normalize_status(v, REASON_CATEGORY)),
    "complaint_text": cb_raw.complaint_text.str.strip(),
    "resolution_status": cb_raw.resolution_status.map(lambda v: normalize_status(v, RESOLUTION_STATUS)),
    "severity": cb_raw.severity.map(lambda v: normalize_status(v, SEVERITY)),
    "channel": cb_raw.channel.map(lambda v: normalize_status(v, CHANNEL)),
    "reported_timestamp": cb_raw.reported_timestamp.map(parse_flexible_date),
    "transaction_timestamp": cb_raw.transaction_timestamp.map(parse_flexible_date),
    "bank_response_timestamp": cb_raw.bank_response_timestamp.map(parse_flexible_date),
})

cb["disputed_amount_missing"] = cb.disputed_amount.isna()
cb["disputed_amount_negative"] = cb.disputed_amount < 0
cb["txn_id_missing"] = cb.txn_id.isna()
cb["reported_ts_missing"] = cb.reported_timestamp.isna()
cb["txn_ts_missing"] = cb.transaction_timestamp.isna()
cb["bank_response_missing"] = cb.bank_response_timestamp.isna()
# impossible ordering - a dispute cannot be raised before the transaction happened
cb["reported_before_txn"] = (cb.reported_timestamp < cb.transaction_timestamp).fillna(False)

for col, mapping in [("severity", SEVERITY), ("resolution_status", RESOLUTION_STATUS), ("channel", CHANNEL), ("reason_code", REASON_CATEGORY)]:
    target = "reason_category" if col == "reason_code" else col
    log(f"{col} values not in mapping: {unmapped_values(cb_raw[col], cb[target])}")

before = len(cb)
cb = cb.drop_duplicates().reset_index(drop=True)
log(f"exact duplicate rows removed (post-normalization): {before - len(cb)}")
cb["complaint_id_duplicate"] = cb.complaint_id.duplicated(keep=False)
log(f"complaint_id still duplicated with differing content: {cb.complaint_id_duplicate.sum()}")

log(f"cleaned rows: {len(cb)}")
for c in ["disputed_amount_missing", "disputed_amount_negative", "txn_id_missing", "reported_ts_missing", "txn_ts_missing", "bank_response_missing", "reported_before_txn"]:
    log(f"  flagged {c}: {cb[c].sum()}")
log(f"  severity distribution: {cb.severity.value_counts(dropna=False).to_dict()}")
log(f"  resolution_status distribution: {cb.resolution_status.value_counts(dropna=False).to_dict()}")
log(f"  reason_category distribution: {cb.reason_category.value_counts(dropna=False).to_dict()}")

cb.to_csv(OUT / "chargebacks_clean.csv", index=False)
cb.head()


=== chargebacks ===
raw rows: 2884
severity values not in mapping: {}
resolution_status values not in mapping: {}
channel values not in mapping: {}
reason_code values not in mapping: {}
exact duplicate rows removed (post-normalization): 84
complaint_id still duplicated with differing content: 0
cleaned rows: 2800
  flagged disputed_amount_missing: 179
  flagged disputed_amount_negative: 220
  flagged txn_id_missing: 77
  flagged reported_ts_missing: 203
  flagged txn_ts_missing: 230
  flagged bank_response_missing: 701
  flagged reported_before_txn: 92
  severity distribution: {'MEDIUM': 1024, 'LOW': 959, 'HIGH': 604, 'CRITICAL': 213}
  resolution_status distribution: {'IN_PROGRESS': 599, 'PENDING_BANK': 458, 'REJECTED': 443, 'CLOSED': 442, 'OPEN': 435, 'RESOLVED': 423}
  reason_category distribution: {'SERVICE_NOT_DELIVERED': 704, 'CUSTOMER_DISPUTE': 376, 'UNAUTHORIZED': 371, 'DUPLICATE_DEBIT': 352, 'ACCOUNT_TAKEOVER': 344, 'FRAUD': 327, 'AMOUNT_MISMATCH': 326}


,complaint_id,txn_id,user_id,merchant_id,disputed_amount,reason_code,reason_category,complaint_text,resolution_status,severity,...,transaction_timestamp,bank_response_timestamp,disputed_amount_missing,disputed_amount_negative,txn_id_missing,reported_ts_missing,txn_ts_missing,bank_response_missing,reported_before_txn,complaint_id_duplicate
0,CBK0002082,TXN00004325,USR97580,MCH1127,NaN,Merchant Not Delivered,SERVICE_NOT_DELIVERED,Customer says amount was debited twice.,CLOSED,CRITICAL,...,2026-01-28 00:00:00,2026-02-10 03:19:10,True,False,False,False,False,False,False,False
1,CBK0001941,TXN00003720,USR54113,MCH3835,414.69,login compromised,ACCOUNT_TAKEOVER,User reports money deducted but merchant denie...,IN_PROGRESS,HIGH,...,2026-02-25 10:24:00,2026-03-12 06:24:00,False,False,False,False,False,False,False,False
2,CBK0001799,TXN00012539,USR17980,MCH3700,7039.00,customer issue,CUSTOMER_DISPUTE,merchant service was not delivered after payment.,OPEN,LOW,...,2026-02-04 00:00:00,2026-03-06 00:00:00,False,False,False,False,False,False,False,False
3,CBK0002465,TXN00017802,USR76148,MCH4534,1303.05,no service,SERVICE_NOT_DELIVERED,Suspicious high-value payment disputed by cust...,REJECTED,HIGH,...,2026-03-30 00:00:00,2026-04-07 00:00:00,False,False,False,False,False,False,False,False
4,CBK0001870,TXN00015944,USR24660,MCH1686,1459.42,merchant service issue,SERVICE_NOT_DELIVERED,Merchant service was not delivered after payment.,IN_PROGRESS,LOW,...,2026-01-09 00:00:00,2026-01-26 00:00:00,False,False,False,False,False,False,False,False


## 5. Join-key coverage and log

Not fixing foreign keys here (that is the Stage 2 data model), just recording how well the cleaned keys line up.

In [6]:
log("")
log("=== join key coverage (informational) ===")
log(f"tx.user_id in kyc: {tx.user_id.isin(kyc.user_id).mean():.1%}")
log(f"tx.merchant_id in merchants: {tx.merchant_id.isin(merchants.merchant_id).mean():.1%}")
log(f"cb.txn_id in tx: {cb.txn_id.isin(tx.txn_id).mean():.1%}  (missing txn_id: {cb.txn_id_missing.sum()})")
log(f"cb.user_id in kyc: {cb.user_id.isin(kyc.user_id).mean():.1%}")
log(f"cb.merchant_id in merchants: {cb.merchant_id.isin(merchants.merchant_id).mean():.1%}")

# unique-id counts are identical before and after dedup, so the gap is in the raw data itself:
# most transaction users/merchants simply have no record in the KYC or merchant master files
log(f"FINDING: only {tx.user_id.isin(kyc.user_id).mean():.1%} of transaction users exist in KYC and "
    f"{tx.merchant_id.isin(merchants.merchant_id).mean():.1%} of transaction merchants exist in the merchant master. "
    f"This is a property of the raw data, not the cleaning: raw unique ids (kyc={kyc.user_id.nunique()}, merchants={merchants.merchant_id.nunique()}) "
    f"match the cleaned tables exactly, so no entity was lost.")

log("")
log("=== summary ===")
for name, raw_n, clean_n in [("upi_transactions", len(tx_raw), len(tx)), ("kyc_records", len(kyc_raw), len(kyc)),
                             ("merchants_master", len(m_raw), len(merchants)), ("chargebacks", len(cb_raw), len(cb))]:
    log(f"{name:18s} raw={raw_n:6d}  cleaned={clean_n:6d}  retained={clean_n/raw_n:.1%}")

(OUT / "cleaning_log.txt").write_text("\n".join(log_lines) + "\n")
print(f"\nwrote {OUT / 'cleaning_log.txt'}")


=== join key coverage (informational) ===


tx.user_id in kyc: 32.4%
tx.merchant_id in merchants: 48.2%


cb.txn_id in tx: 93.1%  (missing txn_id: 77)


cb.user_id in kyc: 31.6%
cb.merchant_id in merchants: 46.4%


FINDING: only 32.4% of transaction users exist in KYC and 48.2% of transaction merchants exist in the merchant master. This is a property of the raw data, not the cleaning: raw unique ids (kyc=28920, merchants=4343) match the cleaned tables exactly, so no entity was lost.

=== summary ===
upi_transactions   raw= 20400  cleaned= 20000  retained=98.0%
kyc_records        raw= 36400  cleaned= 28920  retained=79.5%
merchants_master   raw=  6210  cleaned=  4343  retained=69.9%
chargebacks        raw=  2884  cleaned=  2800  retained=97.1%

wrote ../data/cleaned/cleaning_log.txt
